# 00 — Environment setup
Run once per Colab session type to verify the stack. Confirms Drive layout, package versions, GPU, and that `src/` imports cleanly.

In [ ]:
# --- Colab bootstrap (same cell in every notebook) ---
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/sandesh20lamichhane/when-should-ids-adapt.git /content/repo 2>/dev/null || (cd /content/repo && git pull)
import sys; sys.path.insert(0, '/content/repo')

from src.config import CFG, git_hash, results_path
CFG.make_dirs()
print('commit:', git_hash())

In [ ]:
!pip -q install scikit-learn==1.5.2 scipy pandas pyarrow matplotlib shap
import sklearn, scipy, pandas, numpy
print('sklearn', sklearn.__version__, '| scipy', scipy.__version__,
      '| pandas', pandas.__version__, '| numpy', numpy.__version__)
!nvidia-smi -L || echo 'no GPU (fine for tabular work)'

In [ ]:
# smoke-test every src module
from src import drift, novelty, trigger, streams, metrics
import numpy as np
rng = np.random.default_rng(0)
ref, cur = rng.normal(size=(2000, 8)), rng.normal(0.3, 1, size=(2000, 8))
print('stage1 screen:', drift.screen_window(ref, cur))
nc = novelty.NoveltyCommittee(seed=0).fit(ref)
print('stage2 window novelty:', round(nc.window_novelty(cur), 3))
print('stage3 decision:', trigger.decide(p_novel=0.8, drift_magnitude=0.4,
      batch_size=200, params=trigger.CostParams()))
print('all modules OK')